# Review 2 — Model 1: Bi-LSTM Pit-Stop Predictor

**Deep Learning-Based Virtual Race Engineer for Real-Time Formula 1 Strategy Optimization**

Dhruv Deepak (23BCE9187) · Akhil V (23BCE8051) · Mohan Simha Varma Dantuluri (23BCE9229) · Surya Prathap Mamillapalli (23BCE7783)

---

## What this notebook shows

The first of the project's three models, trained end to end on real Formula 1 telemetry, with the code explained, the results interpreted, and the accuracy graph, loss graph and confusion matrix produced from the held-out test set.

## The task

> Given the last **10 laps** of telemetry for one driver, predict whether that driver will **pit within the next 3 laps**.

This is a binary sequence-classification problem. All three models in the project solve this same task, so their accuracy scores and confusion matrices are directly comparable — that comparison is the Review 3 deliverable.

## Why a 3-lap horizon rather than "pits on this lap"

A driver pits only 2–3 times in a ~55-lap race, so *"pits on this exact lap"* is about **5% positive**. A model that always answers "no pit" would score 95% accuracy and be completely useless. Two things follow, and both shape everything below:

1. The label is widened to a 3-lap window, which lifts the positive rate to a trainable ~13%.
2. **Accuracy is not the headline metric.** We report precision, recall, F1 and PR-AUC alongside it, and we always show the "never pit" baseline for contrast.

## 1. Setup

All pipeline logic lives in `src/` as importable modules rather than in notebook cells. The same code produced the dataset and trains all three models, so nothing here can drift out of sync with what is committed to the repository.

In [ ]:
import sys, json, time, warnings
from pathlib import Path

# Make the project package importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import keras
import matplotlib.pyplot as plt

from src import config
from src.models import architectures, train as train_mod
from src.evaluation import metrics as M, plots

config.ensure_dirs()
config.set_seed()          # seeds python / numpy / tensorflow
plots.apply_style()

print("keras     ", keras.__version__)
print("project   ", config.PROJECT_ROOT)
print("seed      ", config.RANDOM_SEED)

## 2. Load the dataset

`src/features/build_dataset.py` has already turned the raw per-race parquet files into sequence tensors. Each sample is a `(10 laps × 55 features)` window ending at lap *t*, labelled with whether a pit stop follows in laps *t+1 … t+3*.

**The split is by season, not by random row.** Laps from the same race are highly correlated — a random split would put near-duplicate laps in both train and test and inflate the score. Seasons 2021–22 train, 2023 validates, 2024 is held out for testing.

In [ ]:
data, meta = train_mod.load_dataset()

X_train, y_train = data["X_train"], data["y_train"]
X_val,   y_val   = data["X_val"],   data["y_val"]
X_test,  y_test  = data["X_test"],  data["y_test"]
feature_names = meta["features"]

summary = pd.DataFrame({
    "split":     ["train", "validation", "test"],
    "seasons":   [config.TRAIN_SEASONS, config.VAL_SEASONS, config.TEST_SEASONS],
    "samples":   [len(X_train), len(X_val), len(X_test)],
    "positives": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "positive %": [round(100*y_train.mean(), 2), round(100*y_val.mean(), 2), round(100*y_test.mean(), 2)],
})
print(f"window: {meta['seq_len']} laps   horizon: {meta['horizon']} laps   features: {len(feature_names)}")
summary

### The 55 input features

Six groups, all computed per driver per lap. The last group is the important one for this project: undercut pressure is a *cross-driver* phenomenon, so each lap carries field-relative context. These hand-built competitor features are the precursor to the learned cross-driver attention in Model 3.

In [ ]:
groups = {
    "Timing & pace":    [f for f in feature_names if f.startswith(("lap_time", "sector", "deg_"))],
    "Speed traps":      [f for f in feature_names if f.startswith("Speed")],
    "Tyre & stint":     [f for f in feature_names if f.startswith(("tyre", "stint", "compound", "is_fresh", "laps_since", "is_pit", "stops"))],
    "Race position":    [f for f in feature_names if f in ("position","grid_position","position_change","lap_number","lap_frac","laps_remaining")],
    "Track status":     [f for f in feature_names if f.startswith("ts_")],
    "Weather":          [f for f in feature_names if f in ("AirTemp","TrackTemp","Humidity","Pressure","WindSpeed","Rainfall")],
    "Telemetry":        [f for f in feature_names if f.startswith("tel_")],
    "Competitor context": [f for f in feature_names if f in ("gap_to_leader","interval_ahead","interval_behind","n_pitted_this_lap","field_mean_tyre_life","tyre_life_vs_field","frac_field_pitted")],
}
for name, cols in groups.items():
    print(f"{name:<20} {len(cols):>2}  {', '.join(cols)}")

### The class imbalance, stated plainly

This is the single most important fact about the problem, and the reason the evaluation below looks the way it does.

In [ ]:
base_rate = y_test.mean()
print(f"Positive rate in the test set : {100*base_rate:.2f}%")
print(f"'Never pit' accuracy          : {100*(1-base_rate):.2f}%   <- what a useless model scores")
print()
print("So an accuracy figure on its own says almost nothing here.")
print("The confusion matrix and PR-AUC are what reveal whether the model works.")

## 3. The model

A stacked bidirectional LSTM.

**Why bidirectional.** The input is a closed 10-lap window that is already in the past at prediction time, not a live stream — so the model is free to read it in both directions. That lets the later laps of the window inform how the earlier ones are interpreted, and a stint's degradation trend reads more clearly backwards than forwards.

**The shape of the stack.** `BiLSTM(64)` returns the full sequence so `BiLSTM(32)` can compress it into a single summary vector per window; a dense layer then maps that to one sigmoid probability. Dropout of 0.3 sits after each recurrent layer.

**The loss.** Weighted binary cross-entropy, with positives up-weighted by the class ratio (≈6.7). Note this is applied as a *loss function*, not via Keras' `class_weight` argument — `class_weight` is not applied to the validation set, which would put the training and validation loss curves on different scales and make the loss graph unreadable.

In [ ]:
pos_weight = train_mod.positive_weight(y_train)
print(f"positive class weight = {pos_weight:.2f}  "
      f"(one upcoming pit stop counts as much as ~{pos_weight:.0f} quiet laps)\n")

model = architectures.build(
    "bilstm",
    input_shape=(X_train.shape[1], X_train.shape[2]),
    pos_weight=pos_weight,
    learning_rate=config.LEARNING_RATE,
)
model.summary()

## 4. Training

Early stopping monitors **validation PR-AUC**, not validation loss. Under class imbalance the loss can keep creeping down while the model quietly gets worse at the thing we actually care about — identifying pit stops. PR-AUC tracks that directly. The best weights are restored when training stops.

Set `RETRAIN = False` to load the weights saved by `python -m src.models.train --model bilstm` instead of training here.

In [ ]:
RETRAIN = True
CKPT = config.MODELS_DIR / "bilstm.keras"

if RETRAIN:
    t0 = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=config.EPOCHS,
        batch_size=config.BATCH_SIZE,
        callbacks=train_mod.make_callbacks("bilstm"),
        verbose=2,
    )
    train_time = time.perf_counter() - t0
    hist = history.history
    print(f"\ntrained {len(hist['loss'])} epochs in {train_time:.1f}s")
else:
    model = keras.models.load_model(CKPT)
    hist = json.load(open(config.METRICS_DIR / "bilstm_metrics.json"))["history"]
    train_time = float("nan")
    print("loaded saved model and history")

## 5. Accuracy and loss graphs

In [ ]:
acc_png  = plots.plot_accuracy(hist, "bilstm", "Model 1 — Bi-LSTM")
loss_png = plots.plot_loss(hist,  "bilstm", "Model 1 — Bi-LSTM")

from IPython.display import Image, display
display(Image(acc_png)); display(Image(loss_png))

**Reading these two graphs.**

* The dashed line marks the epoch early stopping restored weights from.
* Training and validation loss are on the *same* scale here, because the class weighting lives inside the loss function rather than in `class_weight`. A validation curve that flattens and then turns upward while the training curve keeps falling is the signature of overfitting; the gap between the two curves at the stopping epoch is the honest measure of generalisation.
* Accuracy is plotted because it was asked for, but keep the 87% "never pit" floor in mind while reading it — the interesting movement is in the metrics below, not here.

## 6. Choosing the decision threshold

The network outputs a probability. Turning that into a yes/no call needs a cut-off, and **0.5 is only correct when the classes are balanced** — which they are not.

We pick the threshold that maximises F1 **on the validation set**, then apply that fixed number to the test set. Tuning on test would be cheating: the test set has to stay untouched for the number to mean anything.

In [ ]:
val_prob  = model.predict(X_val,  batch_size=512, verbose=0).ravel()
test_prob = model.predict(X_test, batch_size=512, verbose=0).ravel()

threshold, val_f1 = M.tune_threshold(y_val, val_prob)
print(f"tuned threshold : {threshold:.3f}   (validation F1 = {val_f1:.4f})")

## 7. Results

Three rows, and the comparison between them is the whole point:

1. **Never-pit baseline** — the trivial model, for scale.
2. **Bi-LSTM at threshold 0.50** — the untuned default.
3. **Bi-LSTM at the tuned threshold** — the reported result.

In [ ]:
baseline     = M.majority_baseline(y_test)
test_default = M.compute(y_test, test_prob, 0.5)
test_tuned   = M.compute(y_test, test_prob, threshold)

rows = []
for label, m in [("Never-pit baseline", baseline),
                 ("Bi-LSTM @ 0.50", test_default),
                 ("Bi-LSTM @ tuned", test_tuned)]:
    rows.append({"model": label,
                 "accuracy": m["accuracy"], "balanced acc": m["balanced_accuracy"],
                 "precision": m["precision"], "recall": m["recall"], "f1": m["f1"],
                 "ROC-AUC": m["roc_auc"], "PR-AUC": m["pr_auc"]})
pd.DataFrame(rows).set_index("model").round(4)

In [ ]:
print(M.format_report("Never-pit baseline (test set)", baseline))
print(M.format_report("Model 1 — Bi-LSTM, tuned threshold (test set)", test_tuned))

## 8. Confusion matrix

Cells show the raw count and that count as a share of its true class. The row percentages are what matter: with an imbalanced test set the "No pit" row dwarfs everything else in absolute terms, and only the normalised view shows whether the model genuinely finds pit stops.

* **Bottom-right** — pit stops correctly anticipated. The number the project exists to maximise.
* **Bottom-left** — missed stops. In a real race these are the expensive ones.
* **Top-right** — false alarms. Cheaper: a strategist who is warned and does not stop loses nothing.

In [ ]:
cm_png = plots.plot_confusion_matrix(
    np.array(test_tuned["confusion_matrix"]), "bilstm", "Model 1 — Bi-LSTM")
display(Image(cm_png))

print(f"Pit stops correctly anticipated : {test_tuned['true_positives']:,} "
      f"of {test_tuned['true_positives'] + test_tuned['false_negatives']:,} "
      f"({100*test_tuned['recall']:.1f}%)")
print(f"False alarms                    : {test_tuned['false_positives']:,}")
print(f"When it calls a stop, it is right {100*test_tuned['precision']:.1f}% of the time")

## 9. Threshold-independent curves

The ROC and precision-recall curves score the model's ranking of laps, independent of where the cut-off is drawn.

**The PR curve is the one to read here.** Its baseline is the positive class rate — what random guessing achieves — so the gap between the curve and that dashed line is the model's real contribution. ROC's diagonal baseline flatters imbalanced problems; PR does not.

In [ ]:
c = M.curves(y_test, test_prob)
roc_png = plots.plot_roc(c["fpr"], c["tpr"], test_tuned["roc_auc"], "bilstm", "Model 1 — Bi-LSTM")
pr_png  = plots.plot_precision_recall(c["recall"], c["precision"], test_tuned["pr_auc"],
                                      float(y_test.mean()), "bilstm", "Model 1 — Bi-LSTM")
display(Image(roc_png)); display(Image(pr_png))

## 10. Summary and what comes next

**What was built.** A complete, reproducible pipeline from the FastF1 API to a trained sequence model: ~90 races of real telemetry downloaded and aggregated, 55 engineered features per driver-lap, season-based splits with no leakage, and a Bi-LSTM that reads a 10-lap window and predicts an upcoming pit stop.

**How to read the result.** Compare the tuned Bi-LSTM row against the never-pit baseline. The baseline wins on raw accuracy and scores zero on every metric that matters; the Bi-LSTM trades some of that accuracy for the ability to actually identify pit stops. That trade is the point, and it is exactly what the confusion matrix makes visible.

**Where this goes.** The data pipeline, splits, loss, threshold procedure and metrics are all fixed and shared. Models 2 and 3 plug into the same registry and are scored identically:

| | Model | Status |
|---|---|---|
| 1 | Bi-LSTM | **this notebook** |
| 2 | CNN–BiLSTM | Phase 4 |
| 3 | Multi-task Transformer with cross-driver attention | Phase 5 |

Review 3 compares all three on accuracy, loss, confusion matrices and computational cost.